<a href="https://colab.research.google.com/github/4nw3rprod/OmniRoute/blob/main/notebooks/abliterated-hosted-model-google-colab-synchronization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Download an uncensored (abliterated) GGUF model to Google Drive
Run once. Needs a Google account; a GPU is **not** required for this step.
Change `REPO` / `QUANT` below to pick another model. Free Drive is 15 GB, so stay under ~10 GB.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:

import torch
import psutil
import shutil

print("GPU:", torch.cuda.get_device_name(0))
print("GPU VRAM:", round(
    torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
), "GB")

print("System RAM:", round(
    psutil.virtual_memory().total / 1024**3, 2
), "GB")

print("Available RAM:", round(
    psutil.virtual_memory().available / 1024**3, 2
), "GB")

print("Disk free:", round(
    shutil.disk_usage("/content").free / 1024**3, 2
), "GB")


GPU: Tesla T4
GPU VRAM: 14.56 GB
System RAM: 12.67 GB
Available RAM: 10.98 GB
Disk free: 112.53 GB


In [ ]:
REPO  = "mlabonne/Meta-Llama-3.1-8B-Instruct-abliterated-GGUF"   # any GGUF repo
QUANT = "q5_k_m"                                                   # q4_k_m = smaller, q8_0 = bigger
DEST  = "/content/drive/MyDrive/models"

import os, shutil
from huggingface_hub import HfApi, hf_hub_download
files = [(f.path, f.size) for f in HfApi().list_repo_tree(REPO) if f.path.endswith(".gguf")]
for p, s in files: print(f"{s/1e9:5.2f} GB  {p}")
fn = next(p for p, _ in files if QUANT in p.lower())
print("\nDownloading", fn)
path = hf_hub_download(REPO, fn, local_dir="/content/models")
os.makedirs(DEST, exist_ok=True)
shutil.copy(path, DEST)
print("Saved to Drive:", f"{DEST}/{fn}", os.path.getsize(f"{DEST}/{fn}"), "bytes")

In [9]:
import torch, psutil, shutil

assert torch.cuda.is_available(), "CUDA GPU not available"

print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")
print("RAM:", round(psutil.virtual_memory().total / 1024**3, 2), "GB")
print("Disk:", round(shutil.disk_usage("/content").free / 1024**3, 2), "GB")

GPU: Tesla T4
VRAM: 14.56 GB
RAM: 12.67 GB
Disk: 112.53 GB


In [ ]:
!apt-get update -qq
!apt-get install -y -qq build-essential cmake git libcurl4-openssl-dev

!git clone https://github.com/ggml-org/llama.cpp /content/llama.cpp

%cd /content/llama.cpp

!cmake -B build \
  -DGGML_CUDA=ON \
  -DCMAKE_CUDA_ARCHITECTURES=75 \
  -DLLAMA_CURL=ON \
  -DCMAKE_BUILD_TYPE=Release

!cmake --build build --config Release -j2 --target llama-server

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Cloning into '/content/llama.cpp'...
remote: Enumerating objects: 167143, done.
remote: Counting objects: 100% (174/174), done.
remote: Compressing objects: 100% (115/115), done.
remote: Total 167143 (delta 85), reused 59 (delta 59), pack-reused 166969 (from 3)
Receiving objects: 100% (167143/167143), 491.42 MiB | 26.67 MiB/s, done.
Resolving deltas: 100% (120826/120826), done.
Updating files: 100% (4605/4605), done.
/content/llama.cpp
-- The C compiler identification is GNU 13.3.0
-- The CXX compiler identification is GNU 13.3.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done

In [ ]:
import os

MODEL_REPO = (
    "tpls/gemma-4-12B-coder-fable5-composer2.5-v1-sft-v5-abliterated-GGUF"
)

QUANT = "Q4_K_M"

MODEL = f"{MODEL_REPO}:{QUANT}"

MODEL_DIR = "/content/models/gemma4-coder"

os.makedirs(MODEL_DIR, exist_ok=True)

print("Model:", MODEL)
print("Storage:", MODEL_DIR)

In [ ]:
import subprocess
import time
import os

SERVER = "/content/llama.cpp/build/bin/llama-server"

# Initial conservative settings for Tesla T4
command = [
    SERVER,
    "-hf", MODEL,
    "--cache-dir", MODEL_DIR,
    "--jinja",
    "--ctx-size", "4096",
    "--n-gpu-layers", "99",
    "--batch-size", "256",
    "--ubatch-size", "128",
    "--flash-attn", "on",
    "--host", "0.0.0.0",
    "--port", "8080",
]

# Avoid launching multiple servers accidentally
if "server_process" in globals():
    if server_process.poll() is None:
        server_process.terminate()
        server_process.wait()

server_process = subprocess.Popen(
    command,
    stdout=open("/content/llama-server.log", "w"),
    stderr=subprocess.STDOUT
)

print("Starting Gemma 4 Coder...")
print("PID:", server_process.pid)

In [ ]:
import requests
import time

url = "http://127.0.0.1:8080/health"

for i in range(180):
    if server_process.poll() is not None:
        print("Server exited. Last logs:")
        print(open("/content/llama-server.log").read()[-5000:])
        raise RuntimeError("Model server failed.")

    try:
        response = requests.get(url, timeout=3)
        if response.status_code == 200:
            print("Gemma 4 Coder is ready!")
            break
    except requests.RequestException:
        pass

    time.sleep(5)
else:
    print(open("/content/llama-server.log").read()[-5000:])
    raise TimeoutError("Model did not become ready.")

In [ ]:
from google.colab import output
from IPython.display import display, HTML, IFrame

proxy_url = output.eval_js(
    "google.colab.kernel.proxyPort(8080)"
)

print("Opening Gemma 4 Coder interface:")

display(HTML(
    f'<a href="{proxy_url}" target="_blank">'
    'Open Gemma 4 Coder Chat</a>'
))

display(IFrame(src=proxy_url, width="100%", height=650))

In [ ]:
import requests
import json

API = "http://127.0.0.1:8080/v1/chat/completions"

payload = {
    "model": "gemma4-coder",
    "messages": [
        {
            "role": "system",
            "content": (
                "You are a senior software engineer. "
                "Write production-quality, maintainable code. "
                "Explain important implementation decisions."
            )
        },
        {
            "role": "user",
            "content": """
            Build a reusable Next.js App Router API endpoint
            using TypeScript and Supabase.

            Requirements:
            - Validate incoming request data.
            - Authenticate the user.
            - Prevent unauthorized access.
            - Return structured JSON responses.
            - Handle errors properly.
            - Include TypeScript types.
            """
        }
    ],
    "temperature": 0.3,
    "max_tokens": 1500
}

response = requests.post(API, json=payload, timeout=300)
response.raise_for_status()

result = response.json()

print(result["choices"][0]["message"]["content"])

In [ ]:
!nvidia-smi